In [1]:
import matplotlib.pyplot as plt
import xarray as xr
from sklearn.metrics import r2_score
from pathlib import Path
import pandas as pd
import os

In [2]:
# ------------ Paths ------------
basins_path = Path("./basins_excluded.txt")
original_caravan_path = "/inputs/data_updated_2/time_series"

In [22]:
def open_zarr(path):
  return xr.open_zarr(
      store=path, chunks=None, storage_options=dict(token='anon')
  )

In [23]:
products = [
    'CPC',
    'IMERG',
    'CHIRPS',
    'ERA5_LAND',
    'CHIRPS_GEFS',
    'HRES',
    'GRAPHCAST',
]
zarr_path_template = 'gs://caravan-multimet/v1.1/{}/timeseries.zarr/'

product_to_dataset = {
    product: open_zarr(zarr_path_template.format(product))
    for product in products
}

In [24]:
# Print summary of the different data products.
for product_name, ds in product_to_dataset.items():
  print(f'{product_name}:\n - Number of basins {len(ds["basin"])}')
  print(f' - Start time {ds["date"].values[0]}')
  print(f' - End time {ds["date"].values[-1]}')
  if "lead_time" in ds.coords:
    print(f' - Number of forecast time steps {len(ds["lead_time"])}')
  bands = "".join([f"   - {str(x)}\n" for x in ds.data_vars])
  print(f" - Bands: \n{bands}")

CPC:
 - Number of basins 22492
 - Start time 1979-01-01T00:00:00.000000000
 - End time 2024-07-31T00:00:00.000000000
 - Bands: 
   - cpc_precipitation

IMERG:
 - Number of basins 22492
 - Start time 2000-06-01T00:00:00.000000000
 - End time 2024-10-31T00:00:00.000000000
 - Bands: 
   - imerg_precipitation

CHIRPS:
 - Number of basins 18655
 - Start time 1981-01-01T00:00:00.000000000
 - End time 2024-07-30T00:00:00.000000000
 - Bands: 
   - chirps_precipitation

ERA5_LAND:
 - Number of basins 22485
 - Start time 1950-01-01T00:00:00.000000000
 - End time 2024-10-31T00:00:00.000000000
 - Bands: 
   - era5land_dewpoint_temperature_2m
   - era5land_potential_evaporation_DEPRECATED
   - era5land_potential_evaporation_FAO_PENMAN_MONTEITH
   - era5land_snow_depth_water_equivalent
   - era5land_surface_net_solar_radiation
   - era5land_surface_net_thermal_radiation
   - era5land_surface_pressure
   - era5land_temperature_2m
   - era5land_total_precipitation
   - era5land_u_component_of_wind_10m

In [25]:
with open(basins_path, "r") as f:
    basins = f.read().splitlines()
# basins

In [26]:
# Which data?
product = "GRAPHCAST"
band = "graphcast_temperature_2m"

# Which basin and time period?
basin = "camelsaus_102101A"
time_period = slice("2023-01", "2023-02")

ds = product_to_dataset[product]
# Only with the .compute() in this line, data is actually loaded into memory.
da = ds[band].sel(basin=basin, date=time_period) #.compute()
da

<xarray.DataArray 'graphcast_temperature_2m' (date: 59, lead_time: 10)> Size: 2kB
[590 values with dtype=float32]
Coordinates:
    basin      <U22 88B 'camelsaus_102101A'
  * date       (date) datetime64[ns] 472B 2023-01-01 2023-01-02 ... 2023-02-28
  * lead_time  (lead_time) timedelta64[ns] 80B 1 days 2 days ... 9 days 10 days

In [27]:
product_and_bands= {
    "CHIRPS": ["chirps_precipitation"],
    "ERA5_LAND": ["era5land_total_precipitation"]}

time_period = slice("1989-01-01", "2008-12-31")

In [28]:

original_vars = [
    "total_precipitation_sum",
    "temperature_2m_max",
    "temperature_2m_min",
    "surface_net_solar_radiation_mean",
    "streamflow"
]

for basin in basins:
    # Loop over products, select bands/basins/time, merge into one dataset
    datasets = []
    for product, bands in product_and_bands.items():
        ds = product_to_dataset[product]
        ds_sel = ds[bands].sel(basin=basin, date=time_period).compute()
        datasets.append(ds_sel)

    # Open original caravan dataset and extract variables
    original = xr.open_dataset(f"{original_caravan_path}/{basin}.nc")
    original_sel = original[original_vars].sel(date=time_period)
    datasets.append(original_sel)

    merged = xr.merge(datasets)

    # Save to NetCDF
    output_path = Path(f"./data/{basin}.nc")
    merged.to_netcdf(output_path)
    print(f"Saved {basin} to {output_path}")


Saved camels_01181000 to data/camels_01181000.nc
Saved camels_01411300 to data/camels_01411300.nc
Saved camels_01440400 to data/camels_01440400.nc
Saved camels_01487000 to data/camels_01487000.nc
Saved camels_01491000 to data/camels_01491000.nc
Saved camels_01591400 to data/camels_01591400.nc
Saved camels_01594950 to data/camels_01594950.nc
Saved camels_01605500 to data/camels_01605500.nc
Saved camels_01606500 to data/camels_01606500.nc
Saved camels_01644000 to data/camels_01644000.nc
Saved camels_01664000 to data/camels_01664000.nc
Saved camels_01667500 to data/camels_01667500.nc
Saved camels_02011400 to data/camels_02011400.nc
Saved camels_02014000 to data/camels_02014000.nc
Saved camels_02016000 to data/camels_02016000.nc
Saved camels_02017500 to data/camels_02017500.nc
Saved camels_02018000 to data/camels_02018000.nc
Saved camels_02028500 to data/camels_02028500.nc
Saved camels_02051500 to data/camels_02051500.nc
Saved camels_02053200 to data/camels_02053200.nc
Saved camels_0205380

In [30]:
nc_file=xr.open_dataset("./data/camels_02011400.nc")
nc_file

<xarray.Dataset> Size: 263kB
Dimensions:                           (date: 7305)
Coordinates:
    basin                             <U15 60B ...
  * date                              (date) datetime64[ns] 58kB 1989-01-01 ....
Data variables:
    chirps_precipitation              (date) float32 29kB ...
    era5land_total_precipitation      (date) float32 29kB ...
    total_precipitation_sum           (date) float32 29kB ...
    temperature_2m_max                (date) float32 29kB ...
    temperature_2m_min                (date) float32 29kB ...
    surface_net_solar_radiation_mean  (date) float32 29kB ...
    streamflow                        (date) float32 29kB ...
Attributes:
    Citation:  Funk, Chris, Pete Peterson, Martin Landsfeld, Diego Pedreros, ...
    License:   To the extent possible under the law, Pete Peterson has waived...
    Product:   CHIRPS
    Released:  2024-11-18
    Sources:   CHIRPS: Rainfall Estimates from Rain Gauge and Satellite Obser...
    Units:     precipitation [mm]
    Version:   1.1

In [31]:
# basin="camels_02011400"
# precip=pd.read_csv(f"./mswep_precip_timeseries/{basin}_precip.csv")
# precip.head()

In [32]:
# precip_da = xr.DataArray(
#     data=precip["precip_mm"].values,
#     dims=["date"],
#     coords={"date": pd.to_datetime(precip["date"]).values},
# )
# nc_file["mswep_precipitation"] = precip_da
# nc_file

In [33]:
for basin in basins:
    try:
        nc_path = f"./data/{basin}.nc"
        ds = xr.open_dataset(nc_path)
        precip = pd.read_csv(f"./mswep_precip_timeseries/{basin}_precip.csv")
        precip_da = xr.DataArray(
            data=precip["precip_mm"].values,
            dims=["date"],
            coords={"date": pd.to_datetime(precip["date"]).values},
        )
        ds["mswep_precipitation"] = precip_da
        ds.to_netcdf(nc_path + ".tmp")
        ds.close()

        os.replace(nc_path + ".tmp", nc_path)
        print(f"Added mswep_precipitation to {basin}")
    except Exception as e:
        print(f"Failed for {basin}: {e}")

Added mswep_precipitation to camels_01181000
Added mswep_precipitation to camels_01411300
Added mswep_precipitation to camels_01440400
Added mswep_precipitation to camels_01487000
Added mswep_precipitation to camels_01491000
Added mswep_precipitation to camels_01591400
Added mswep_precipitation to camels_01594950
Added mswep_precipitation to camels_01605500
Added mswep_precipitation to camels_01606500
Added mswep_precipitation to camels_01644000
Added mswep_precipitation to camels_01664000
Added mswep_precipitation to camels_01667500
Added mswep_precipitation to camels_02011400
Added mswep_precipitation to camels_02014000
Added mswep_precipitation to camels_02016000
Added mswep_precipitation to camels_02017500
Added mswep_precipitation to camels_02018000
Added mswep_precipitation to camels_02028500
Added mswep_precipitation to camels_02051500
Added mswep_precipitation to camels_02053200
Added mswep_precipitation to camels_02053800
Added mswep_precipitation to camels_02056900
Added mswe

In [34]:
xr.open_dataset("./data/camels_02011400.nc")

<xarray.Dataset> Size: 321kB
Dimensions:                           (date: 7305)
Coordinates:
    basin                             <U15 60B ...
  * date                              (date) datetime64[ns] 58kB 1989-01-01 ....
Data variables:
    chirps_precipitation              (date) float32 29kB ...
    era5land_total_precipitation      (date) float32 29kB ...
    total_precipitation_sum           (date) float32 29kB ...
    temperature_2m_max                (date) float32 29kB ...
    temperature_2m_min                (date) float32 29kB ...
    surface_net_solar_radiation_mean  (date) float32 29kB ...
    streamflow                        (date) float32 29kB ...
    mswep_precipitation               (date) float64 58kB ...
Attributes:
    Citation:  Funk, Chris, Pete Peterson, Martin Landsfeld, Diego Pedreros, ...
    License:   To the extent possible under the law, Pete Peterson has waived...
    Product:   CHIRPS
    Released:  2024-11-18
    Sources:   CHIRPS: Rainfall Estimates from Rain Gauge and Satellite Obser...
    Units:     precipitation [mm]
    Version:   1.1

In [3]:
import xarray as xr
from pathlib import Path
import glob

rename_map = {
    "prcp_chirps_mm_day": "chirps_precipitation",
    "prcp_mswep_mm_day": "mswep_precipitation",
    "srad_W_m2": "surface_net_solar_radiation_mean",
    "tmax_C": "temperature_2m_max",
    "tmin_C": "temperature_2m_min",
    "QObs_mm_d": "streamflow",
}

nc_files = sorted(glob.glob("./data/time_series/CAMELS_UY_*.nc"))
print(f"Found {len(nc_files)} files")

for f in nc_files:
    ds = xr.open_dataset(f)
    # Only rename variables that exist in this file
    to_rename = {k: v for k, v in rename_map.items() if k in ds.data_vars}
    if to_rename:
        ds = ds.rename(to_rename)
        ds.to_netcdf(f + ".tmp")
        Path(f + ".tmp").rename(f)
        print(f"Renamed {list(to_rename.keys())} in {Path(f).name}")
    else:
        print(f"No matching vars in {Path(f).name}")
    ds.close()

print("Done!")

Found 11 files
Renamed ['prcp_chirps_mm_day', 'prcp_mswep_mm_day', 'srad_W_m2', 'tmax_C', 'tmin_C', 'QObs_mm_d'] in CAMELS_UY_10.nc
Renamed ['prcp_chirps_mm_day', 'prcp_mswep_mm_day', 'srad_W_m2', 'tmax_C', 'tmin_C', 'QObs_mm_d'] in CAMELS_UY_11.nc
Renamed ['prcp_chirps_mm_day', 'prcp_mswep_mm_day', 'srad_W_m2', 'tmax_C', 'tmin_C', 'QObs_mm_d'] in CAMELS_UY_15.nc
Renamed ['prcp_chirps_mm_day', 'prcp_mswep_mm_day', 'srad_W_m2', 'tmax_C', 'tmin_C', 'QObs_mm_d'] in CAMELS_UY_16.nc
Renamed ['prcp_chirps_mm_day', 'prcp_mswep_mm_day', 'srad_W_m2', 'tmax_C', 'tmin_C', 'QObs_mm_d'] in CAMELS_UY_2.nc
Renamed ['prcp_chirps_mm_day', 'prcp_mswep_mm_day', 'srad_W_m2', 'tmax_C', 'tmin_C', 'QObs_mm_d'] in CAMELS_UY_3.nc
Renamed ['prcp_chirps_mm_day', 'prcp_mswep_mm_day', 'srad_W_m2', 'tmax_C', 'tmin_C', 'QObs_mm_d'] in CAMELS_UY_5.nc
Renamed ['prcp_chirps_mm_day', 'prcp_mswep_mm_day', 'srad_W_m2', 'tmax_C', 'tmin_C', 'QObs_mm_d'] in CAMELS_UY_6.nc
Renamed ['prcp_chirps_mm_day', 'prcp_mswep_mm_day', '

In [4]:
xr.open_dataset("./data/time_series/CAMELS_UY_2.nc")

<xarray.Dataset> Size: 498kB
Dimensions:                           (date: 11322)
Coordinates:
  * date                              (date) datetime64[ns] 91kB 1989-01-01 ....
Data variables:
    temperature_2m_min                (date) float32 45kB ...
    temperature_2m_max                (date) float32 45kB ...
    surface_net_solar_radiation_mean  (date) float32 45kB ...
    prcp_mm_day                       (date) float32 45kB ...
    streamflow                        (date) float64 91kB ...
    prcp_gauge_mm_day                 (date) float32 45kB ...
    mswep_precipitation               (date) float32 45kB ...
    chirps_precipitation              (date) float32 45kB ...
Attributes:
    precip_update:  Gauge, MSWEP and CHIRPS added where available